In [2]:
import os
import pickle
from pathlib import Path
import pandas as pd

FILE_PATH = os.path.dirname(os.path.abspath("."))
os.chdir(FILE_PATH)
from src.bm25 import BM25Search
from src.semantic import SemanticSearch
from src.hybrid import HybridSearch
from src.download_data import download_data
os.chdir(f"{FILE_PATH}/notebooks")

download_data()

CATEGORY = "Appliances"
PROCESSED_DATA_DIR = Path("../data/processed")
with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_product_documents.pkl", "rb") as f:
    documents = pickle.load(f)

with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_doc_ids.pkl", "rb") as f:
    doc_ids = pickle.load(f)

import duckdb

PROCESSED_DATA_DIR = Path("../data/processed")
product_data_file = "Appliances_products.parquet"

c2 = duckdb.connect()
products = c2.execute(f"SELECT * FROM read_parquet('{PROCESSED_DATA_DIR}/{product_data_file}')").df()

/Users/harrisonlee/miniforge3/envs/dsci575-project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: /Users/harrisonlee/Code/ubc-mds/Block 6/DSCI575/DSCI_575_project_hli76_wnsong/src
Review data for Appliances already downloaded
Meta data for Appliances already downloaded
Merged data for Appliances is ready
Products Data for Appliances is ready
Document id for Appliances is ready
Document id for Appliances is ready


In [3]:
print(products.keys())

Index(['parent_asin', 'product_title', 'main_category', 'store', 'price',
       'avg_rating', 'reviews', 'review_titles', 'helpful_votes'],
      dtype='str')


In [4]:
top_k = 10

bm25 = BM25Search(documents)
semantic = SemanticSearch(documents)
hybrid = HybridSearch(
    bm25=bm25, semantic=semantic, alpha=0.5, top_k_candidates=top_k + 100
)

load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7200.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done


## Create RAG Pipeline using Semantic Retriever

```pseudocode
def pipeline(query):
    retriever = SemanticSearch(Documents)
    retrieved_products = retriever.search(query)
    context = build_docs(retrieved_products)
    prompt = build_prompt(query, context)
    return format_output(llm(prompt))
```

In [5]:
query = "Best container for my food that needs to be cold"

In [ ]:
def build_context(results):
    context = ""
    for i, (index, score) in enumerate(results):
        product_asin = doc_ids[index]
        product_context = documents[index]
        # print(f"{i+1}. ({score:.3f}) {product.product_title.values[0]}")
        context += f"""
parent_asin: {product_asin}
{product_context}

"""
    return context

DEFAULT_SYSTEM_PROMPT = """
Instructions:
- You are a helpful Amazon shopping assistant.
- You must answer the question using ONLY the following context (real product reviews with helpful votes and the metadata for the products).
- Always cite the product ASIN when possible.
- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.
"""

def build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT):
    return f"""
{system_prompt}

---------

Context: 
{context}

---------

Question:
{query}

"""

def RAG_pipeline(retriever, query, llm):
    results = retriever.search(query, top_k=5)
    context = build_context(results)
    prompt = build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT)

    return prompt


print(RAG_pipeline(semantic, query))



Instructions:
- You are a helpful Amazon shopping assistant.
- You must answer the question using ONLY the following context (real product reviews with helpful votes and the metadata for the products).
- Always cite the product ASIN when possible.
- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.


---------

Context: 

parent_asin: B08Y5JGHTB
Title: KUPPET 3.5 Cu Ft Compact Chest Freezer, with Flip-up Lid, with Removable Basket, Adjustable thermostat, 7 Temperature Setting, for Apartment, Garage, Restaurant (White) KUPPET 3.5 Cu Ft Compact Chest Freezer, with Flip-up Lid, with Removable Basket, Adjustable thermostat, 7 Temperature Setting, for Apartment, Garage, Restaurant (White)
Category: Appliances
Store: KUPPET
Price: 169.99
Average Rating: 4.5

Reviews:
- (2 votes) Excellent for its price: Pretty good unit for its price. 9f cou